In [12]:
pip install numpy==1.25.2 --force-reinstall


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 1.1 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.23.5
    Uninstalling numpy-1.23.5:
      Successfully uninstalled numpy-1.23.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.17.0 requires keras>=3.2.0, but you have keras 2.12.0 which is incompatible.
tensorflow 2.17.0 requires tensorboard<2.18,>=2.17, but you have tensorboard 2.12.3 which is incompatible.
tensorflow-macos 2.12.0 requires numpy<1.24,>=1.22, but you have numpy 1.25.2 which is incompatible.
scikeras 0.13.0 requires keras>=3.2.0, but you have keras 2.12.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from lightfm.data import Dataset

/opt/anaconda3/lib/python3.11/site-packages/lightfm/_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


### Loading behaviours.tsv and converting to interactions matrix

In [4]:
# 1) Load MIND-small behaviors
beh = pd.read_csv(
    'MINDsmall_train/behaviors.tsv',
    sep='\t', header=None,
    names=['impression_id','user_id','timestamp','history','impressions'],
    dtype=str
)
# split the space-separated click history into lists
beh['history'] = beh['history'].fillna('').str.split()

In [6]:
# expand into one row per click
rows = []
for uid, hist in zip(beh['user_id'], beh['history']):
    for aid in hist:
        rows.append({'user_id': uid, 'article_id': aid, 'clicked': 1})
interactions_df = pd.DataFrame(rows)


In [16]:
print(f" Loaded {len(interactions_df):,} click events")
print(f"  • {interactions_df['user_id'].nunique():,} unique users")
print(f"  • {interactions_df['article_id'].nunique():,} unique articles")


 Loaded 5,107,639 click events
  • 49,108 unique users
  • 33,195 unique articles


In [10]:
dataset = Dataset()
dataset.fit(
    users=interactions_df['user_id'].unique(),
    items=interactions_df['article_id'].unique()
)

In [12]:
(interactions, _) = dataset.build_interactions(
    interactions_df[['user_id','article_id']].itertuples(index=False, name=None)
)

In [18]:
print(f" Interaction matrix: {interactions.shape[0]} users × {interactions.shape[1]} items")

 Interaction matrix: 49108 users × 33195 items


### Loading news.tsv and preprocessing (building item features through TF-IDF on titles and OHE on vertical categories)

In [4]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix, hstack

In [6]:
# 1) Load news metadata
news = pd.read_csv(
    'MINDsmall_train/news.tsv',
    sep='\t', header=None,
    names=['newid','vertical','subvertical','title','abstract','url','ent_title','ent_abstract'],
    dtype=str
).set_index('newid')


In [8]:
print(news['vertical'].nunique())

17


In [30]:
# 2) Align rows to LightFM’s item order
item_map    = dataset.mapping()[2]                     # dict: article_id → row_idx
article_ids = [aid for aid,_ in sorted(item_map.items(), key=lambda x: x[1])]
news        = news.reindex(article_ids).fillna('')     # now news.loc[aid] matches row i in dataset


In [35]:
# 3) TF-IDF on article titles
tfidf = TfidfVectorizer(max_features=2000, stop_words='english')
tfidf_matrix = tfidf.fit_transform(news['title'])       # shape: (n_items, 2000)

max_features = 2000 : Build a vocabulary of the top 2000 most important words across all titles, based on their term frequency-inverse document frequency (TF-IDF) scores.

In [33]:
# 4) One-hot encode vertical categories
vert_ohe = pd.get_dummies(news['vertical'], prefix='vert')
vert_matrix = csr_matrix(vert_ohe.values)               # shape: (n_items, #unique_verticals)


In [37]:
item_features = hstack([vert_matrix, tfidf_matrix], format='csr')


In [39]:
print(f"Built item_features with shape {item_features.shape}")
print(f"  • {vert_matrix.shape[1]} vertical features")
print(f"  • {tfidf_matrix.shape[1]} TF-IDF title features")
print(f"  • Total nonzeros: {item_features.nnz:,}")

Built item_features with shape (33195, 2016)
  • 16 vertical features
  • 2000 TF-IDF title features
  • Total nonzeros: 191,466


### Interaction matrices for both train and val datasets

In [45]:
beh_dev = pd.read_csv(
    'MINDsmall_dev/behaviors.tsv',
    sep='\t', header=None,
    names=['impression_id','user_id','timestamp','history','impressions'],
    dtype=str
)
beh_dev['history'] = beh_dev['history'].fillna('').str.split()

In [49]:
def expand(df):
    rows = []
    for uid, hist in zip(df['user_id'], df['history']):
        for aid in hist:
            rows.append((uid, aid))
    return pd.DataFrame(rows, columns=['user_id','article_id'])

train_df = expand(beh)
val_df   = expand(beh_dev)

In [51]:
print(f"Train clicks: {len(train_df):,}; Val clicks: {len(val_df):,}")


Train clicks: 5,107,639; Val clicks: 2,362,514


In [53]:
# 3) Union of all users and items
all_users = pd.Index(train_df['user_id']).union(val_df['user_id'])
all_items = pd.Index(train_df['article_id']).union(val_df['article_id'])

In [55]:
# 4) Fit the Dataset on the union
dataset = Dataset()
dataset.fit(users=all_users, items=all_items)

#### Reconstructing item features using new dataset.mapping()[2]

In [67]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix, hstack

# 1) Reload news metadata
news = pd.read_csv(
    'MINDsmall_train/news.tsv',
    sep='\t', header=None,
    names=['newid','vertical','subvertical','title','abstract','url','ent_title','ent_abstract'],
    dtype=str
).set_index('newid')

# 2) Get the new mapping and reorder news to match it
item_map    = dataset.mapping()[2]  # article_id → row_idx (now 44 908 items)
article_ids = [aid for aid,_ in sorted(item_map.items(), key=lambda x: x[1])]
news        = news.reindex(article_ids).fillna('')

# 3) TF-IDF on titles
tfidf = TfidfVectorizer(max_features=2000, stop_words='english')
tfidf_matrix = tfidf.fit_transform(news['title'])  # (n_items, 2000)

# 4) One-hot verticals
vert_ohe     = pd.get_dummies(news['vertical'], prefix='vert')
vert_matrix  = csr_matrix(vert_ohe.values)        # (n_items, n_vert)

# 5) Stack into final item_features
item_features = hstack([vert_matrix, tfidf_matrix], format='csr')

print(f"Rebuilt item_features → shape: {item_features.shape}")


Rebuilt item_features → shape: (44908, 2017)


In [75]:
# 1) Build a set of all train pairs
train_pairs = set(zip(train_df['user_id'], train_df['article_id']))

# 2) Filter val_df to remove any overlap
val_df_filtered = val_df[
    ~val_df.apply(lambda r: (r['user_id'], r['article_id']) in train_pairs, axis=1)
].reset_index(drop=True)

print(f"Filtered out {len(val_df) - len(val_df_filtered):,} overlapping clicks; {len(val_df_filtered):,} remain.")

# 3) Rebuild the validation interactions
val_interactions, _ = dataset.build_interactions(
    val_df_filtered[['user_id','article_id']].itertuples(index=False, name=None)
)



Filtered out 308,776 overlapping clicks; 2,053,738 remain.


In [77]:
# 5) Build interactions for train and val
train_interactions, _ = dataset.build_interactions(
    train_df[['user_id','article_id']].itertuples(index=False, name=None)
)

print(f"Interaction matrices built:")
print(f"   Train: {train_interactions.shape[0]} users × {train_interactions.shape[1]} items")
print(f"   Val:   {val_interactions.shape[0]} users × {val_interactions.shape[1]} items")

Interaction matrices built:
   Train: 91935 users × 44908 items
   Val:   91935 users × 44908 items


### Model Training

In [61]:
from lightfm import LightFM
from lightfm.evaluation import precision_at_k


In [63]:
model = LightFM(
    no_components=50,
    loss='warp'
)

In [70]:
model.fit(
    train_interactions,
    item_features=item_features,
    epochs=20,
    num_threads=4
)

In [71]:
# 3c) Evaluate Precision@10 on train and val
train_prec = precision_at_k(
    model,
    train_interactions,
    item_features=item_features,
    k=10
).mean()


In [79]:
val_prec = precision_at_k(
    model,
    val_interactions,
    train_interactions=train_interactions,  # mask out seen train items
    item_features=item_features,
    k=10
).mean()


In [80]:
print(f"Precision@10 → train: {train_prec:.4f}, validation: {val_prec:.4f}")

Precision@10 → train: 0.0714, validation: 0.0488
